# Inverse problem training on a 7 compartment epidemic model: Forward simulation, Neural ODEs and UDEs

## Introduction
In this tutorial, we will look at a 7 compartment epidemic model consisting of the following compartments: susceptible, exposed, infected, removed, total population, severe cases and deaths, and cumulative cases. The dynamics also include a time varying contact rate, which falls as the severe cases and deaths rise.

## Procedure
- First, we simulate 21 days of data from the 7 compartment model. This is a forward simulation using the `Vern7` ODE solver from the `OrdinaryDiffEq.jl` Julia library.
- We then define and train a Neural ODE on the exposed, infected and removed of this simulation.
- We then turn towards a Universal Differential Equation (UDE) formulation. We assume that the time varying nature of the contact rate is not known. Hence, we replace this term with a neural network. Using this UDE, we again train the model on the same data.
- Finally we extrapolate both models to 60 days. Through the Neural ODE and the UDE training, we shed light on the optimizer interface of Optimization.jl and SciMLSensitivity.jl.

## Libraries

In [ ]:
Pollis.packages("Lux", "OrdinaryDiffEq", "SciMLSensitivity", "Optimization", "OptimizationOptimisers", "OptimizationOptimJL", "ComponentArrays", "Mooncake", "Plots")

In [ ]:
using Lux, OrdinaryDiffEq, SciMLSensitivity, Optimization, OptimizationOptimisers, OptimizationOptimJL, ComponentArrays, Mooncake, Random, Plots

rng = Xoshiro(1)

## Data generation: forward simulation
The parameters are F, the imported cases, beta_0, alpha, kappa, mu, sigma, gamma, d and lambda. The contact rate is beta(t) = beta_0 (1 - alpha) (1 - D/N)^kappa.

In [ ]:
contact(beta0, D, N, kappa, alpha) = beta0 * (1 - alpha) * (1 - D / N)^kappa

function corona!(du, u, p, t)
    S, E, I, R, N, D, C = u
    F, beta0, alpha, kappa, mu, sigma, gamma, d, lambda = p
    beta = contact(beta0, D, N, kappa, alpha)
    du[1] = -beta0 * S * F / N - beta * S * I / N - mu * S         # susceptible
    du[2] = beta0 * S * F / N + beta * S * I / N - (sigma + mu) * E  # exposed
    du[3] = sigma * E - (gamma + mu) * I                           # infected
    du[4] = gamma * I - mu * R                                     # removed (recovered + dead)
    du[5] = -mu * N                                                # total population
    du[6] = d * gamma * I - lambda * D                             # severe, critical cases, and deaths
    du[7] = sigma * E                                              # cumulative cases
end

S0 = 14e6
u0 = [0.9 * S0, 0.0, 0.0, 0.0, S0, 0.0, 0.0]
p_ = [10.0, 0.5944, 0.4239, 1117.3, 0.02, 1 / 3, 1 / 5, 0.2, 1 / 11.2]
R0 = p_[2] / p_[7] * p_[6] / (p_[6] + p_[5])
println("R0 = ", round(R0; digits = 2))

tspan = (0.0, 21.0)
prob = ODEProblem(corona!, u0, tspan, p_)
solution = solve(prob, Vern7(); abstol = 1e-12, reltol = 1e-12, saveat = 1)

tspan2 = (0.0, 60.0)
solution_extrapolate = solve(remake(prob; tspan = tspan2), Vern7(); abstol = 1e-12, reltol = 1e-12, saveat = 1)

# The data: the exposed, infected and removed, with 2% noise
tsdata = Array(solution)
noisy_data = tsdata .* (1 .+ 0.02 .* randn(rng, size(tsdata)))
observed = noisy_data[2:4, :]

# The counts are in the hundreds: the networks work on counts divided by this scale
scale = 100.0

losses = Float64[]
callback = function (state, l)
    push!(losses, l)
    length(losses) % 100 == 0 && println("Loss after ", length(losses), " iterations: ", l)
    return false
end

function extrapolation_plot(estimate, title)
    plt = plot(; legend = :topleft, title = title, xlabel = "Day", ylabel = "People")
    for (row, name, colour) in ((2, "Exposed", :orange), (3, "Infected", :red), (4, "Removed", :blue))
        scatter!(plt, solution_extrapolate.t, solution_extrapolate[row, :]; color = colour, ms = 3, label = "True " * name)
        plot!(plt, solution_extrapolate.t, estimate[row - 1, :]; color = colour, lw = 3, label = "Estimated " * name)
    end
    vline!(plt, [21]; color = :black, lw = 2, label = "Training Data End")
    return plt
end

## Defining and training a Neural ODE
The Neural ODE models the exposed, infected and removed alone, with a network for their derivatives: it is given no epidemiology.

In [ ]:
ann_node = Chain(Dense(3 => 32, tanh), Dense(32 => 32, tanh), Dense(32 => 3))
p1, st1 = Lux.setup(rng, ann_node)

dudt_node(x, p, t) = first(ann_node(x, p, st1))
x0 = observed[:, 1] ./ scale
prob_node = ODEProblem(dudt_node, x0, tspan)

predict_node(theta, ts = solution.t) = Array(solve(prob_node, Tsit5(); p = theta, saveat = ts, tspan = (0.0, last(ts)), abstol = 1e-6, reltol = 1e-6))
loss_node(theta, _) = sum(abs2, observed ./ scale .- predict_node(theta))

empty!(losses)
optf = OptimizationFunction(loss_node, Optimization.AutoMooncake(; config = nothing))
optprob = OptimizationProblem(optf, ComponentArray(p1))
res1 = solve(optprob, Adam(0.05); callback = callback, maxiters = 300)
res2 = solve(remake(optprob; u0 = res1.u), Optim.BFGS(; initial_stepnorm = 0.01); callback = callback, maxiters = 200)
println("Neural ODE loss: ", losses[end])

# Extrapolate out
_sol_node = predict_node(res2.u, solution_extrapolate.t) .* scale
extrapolation_plot(_sol_node, "Neural ODE Extrapolation")

We see that the Neural ODE extrapolation is poor. This is because a Neural ODE does not incorporate any information regarding the physics of the system. Hence, a Neural ODE is generally poor at forecasting. We now turn towards a UDE where we retain the physics of the system, and replace the missing piece with a neural network.

## Defining and training a UDE
The UDE keeps all 7 equations and every known parameter, and replaces the contact rate, the term that is not known, with a network of S/N, I and D/N: the network gives beta(t) S/N, the rate at which each infected person infects.

In [ ]:
ann = Chain(Dense(3 => 32, tanh), Dense(32 => 1))
p2, st2 = Lux.setup(rng, ann)

function dudt_(u, p, t)
    S, E, I, R, N, D, C = u
    F, beta0, alpha, kappa, mu, sigma, gamma, d, lambda = p_
    z = abs(first(first(ann([S / N, I / scale, D / N * 1e4], p, st2)))) * I
    return [
        -beta0 * S * F / N - z - mu * S,          # susceptible
        beta0 * S * F / N + z - (sigma + mu) * E,  # exposed
        sigma * E - (gamma + mu) * I,             # infected
        gamma * I - mu * R,                       # removed (recovered + dead)
        -mu * N,                                  # total population
        d * gamma * I - lambda * D,               # severe, critical cases, and deaths
        sigma * E,                                # cumulative cases
    ]
end

prob_nn = ODEProblem(dudt_, u0, tspan)
predict_ude(theta, ts = solution.t) = Array(solve(prob_nn, Tsit5(); p = theta, saveat = ts, tspan = (0.0, last(ts)), abstol = 1e-6, reltol = 1e-6))
loss_ude(theta, _) = sum(abs2, (observed .- predict_ude(theta)[2:4, :]) ./ scale)

empty!(losses)
optf = OptimizationFunction(loss_ude, Optimization.AutoMooncake(; config = nothing))
optprob = OptimizationProblem(optf, ComponentArray(p2))
res1 = solve(optprob, Adam(0.05); callback = callback, maxiters = 300)
res2_ude = solve(remake(optprob; u0 = res1.u), Optim.BFGS(; initial_stepnorm = 0.01); callback = callback, maxiters = 200)
println("UDE loss: ", losses[end])

# Extrapolate out
_sol_uode = predict_ude(res2_ude.u, solution_extrapolate.t)[2:4, :]
extrapolation_plot(_sol_uode, "UDE Extrapolation")

## The learned contact rate
The network's beta(t) S/N against the true one, over the 60 days.

In [ ]:
truth = Array(solution_extrapolate)
learned = predict_ude(res2_ude.u, solution_extrapolate.t)
true_rate = [contact(p_[2], u[6], u[5], p_[4], p_[3]) * u[1] / u[5] for u in eachcol(truth)]
learned_rate = [abs(first(first(ann([u[1] / u[5], u[3] / scale, u[6] / u[5] * 1e4], res2_ude.u, st2)))) for u in eachcol(learned)]
plot(solution_extrapolate.t, true_rate; label = "True", lw = 2, ls = :dash, color = :black, xlabel = "Day", ylabel = "beta(t) S/N")
plot!(solution_extrapolate.t, learned_rate; label = "UDE", lw = 2, color = :red)
vline!([21]; color = :black, label = "Training Data End")

We can see that the UDE forecasting is much better than the Neural ODE! This can be attributed to the system physics being retained in the UDE framework: only one term has to be learned, and it is learned as a rate, which the model then carries forward.

Through both the Neural ODE and UDE training procedures, we shed light on how the optimizer interface (Optimization.jl and SciMLSensitivity.jl) acts as the interface between equation solvers and automatic differentiation (AD) tooling.